# Graph RAG Explorer

Interaktivni notebook za istrazivanje celog RAG + Graph Expansion pipeline-a korak po korak.

**Preduslovi:** Pokreni iz `chatbot-api/` direktorijuma sa pristupom Weaviate i Neo4j serverima.

## 0. Setup i konekcije

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

# Svi sekreti se učitavaju iz ../.env — ne hardkoduj ih ovde.
from urllib.parse import urlparse
from starlette.config import Config
_cfg = Config(os.getenv('ENV_FILE', '../.env'))

NEO4J_URI = _cfg('NEO4J_URI', cast=str, default='bolt://localhost:7687')
NEO4J_USER = _cfg('NEO4J_USER', cast=str, default='neo4j')
NEO4J_PASS = _cfg('NEO4J_PASS', cast=str)
NEO4J_DATABASE = _cfg('NEO4J_DATABASE_DIPLO', cast=str, default='weaviatediplo')  # ili 'weaviatedw' za dig.watch

_wv_url = _cfg('WV_CLIENT_URL', cast=str)
_wv_parsed = urlparse(_wv_url)
WEAVIATE_HOST = _wv_parsed.hostname or '127.0.0.1'
WEAVIATE_PORT = _wv_parsed.port or 8080
WEAVIATE_GRPC_PORT = _cfg('WV_GRPC_PORT', cast=int, default=50051)
WEAVIATE_API_KEY = _cfg('WV_KEY', cast=str)

In [2]:
# ── Weaviate konekcija ──
import weaviate

wv_client = weaviate.connect_to_local(
    host=WEAVIATE_HOST,
    port=WEAVIATE_PORT,
    grpc_port=WEAVIATE_GRPC_PORT,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
    additional_config=weaviate.classes.init.AdditionalConfig(
        timeout=weaviate.classes.init.Timeout(init=30)
    ),
)
print(f"Weaviate connected: {wv_client.is_ready()}")
print(f"Collections: {[c for c in wv_client.collections.list_all().keys()]}")

Weaviate connected: True
Collections: ['DiploChunk', 'DiploChunk_contextual', 'DiploDocument', 'DiploDocument_contextual', 'DiploHeading', 'DiploParagraph', 'DiploParagraph_contextual', 'DiploTurn', 'DiploTurn_contextual']


In [3]:
# ── Neo4j konekcija ──
from app.ai.ai_services.neo4j_graph_client import Neo4jGraphClient

neo4j_client = Neo4jGraphClient(
    uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASS,
    database_diplo=NEO4J_DATABASE,
)
await neo4j_client.connect()
healthy = await neo4j_client.health_check()
print(f"Neo4j healthy: {healthy}")

Neo4j healthy: True


## 1. Weaviate Retrieval

Hybrid search (vektor + BM25) na `DiploChunk_contextual` kolekciji.

In [4]:
# ── Postavi pitanje ──
QUESTION = "What is AI governance and how does Diplo approach it?"

In [5]:
from weaviate.classes.query import HybridFusion, Filter, MetadataQuery
from app.ai.ai_services.embeddings import TEIEmbeddings
from app.core.config import LOCAL_EMBEDDING_URL, LOCAL_EMBEDDING_KEY

embeddings = TEIEmbeddings(url=LOCAL_EMBEDDING_URL, api_key=LOCAL_EMBEDDING_KEY)
query_vector = embeddings.embed_query(QUESTION)
print(f"Query embedded: {len(query_vector)} dimensions")

collection = wv_client.collections.get("DiploChunk_contextual")

results = collection.query.hybrid(
    query=QUESTION,
    vector=query_vector,
    alpha=0.75,
    fusion_type=HybridFusion.RELATIVE_SCORE,
    limit=30,
    filters=Filter.by_property("chunk_level").equal("sentence")
        & Filter.by_property("visibility").not_equal("private"),
    query_properties=["sentence"],
    return_metadata=["score"],
)

print(f"\nRetrieved {len(results.objects)} sentences")
print("\nTop 10:")
for i, obj in enumerate(results.objects[:10]):
    p = obj.properties
    score = obj.metadata.score if obj.metadata else 0
    url = (p.get('link') or '')[:60]
    sent = (p.get('sentence') or '')[:100]
    print(f"  [{i+1}] score={score:.4f} | {url}")
    print(f"       {sent}...")

Query embedded: 1024 dimensions

Retrieved 30 sentences

Top 10:
  [1] score=0.7500 | https://www.diplomacy.edu/diplo-ai-agents/
       Much of Diplo's work centeres on governance of AI, internet, and overall tech developmetns....
  [2] score=0.6478 | https://www.diplomacy.edu/diplo-news/issue479/
       Diplo’s Dr Jovan Kurbalija explains the layers of AI governance, which include hardware, data, algor...
  [3] score=0.5103 | https://www.diplomacy.edu/diplo-ai-agents/
       By seamlessly blending Diplo’s decades of expert knowledge in diplomacy and global governance with t...
  [4] score=0.4918 | https://www.diplomacy.edu/blog/diplos-crystal-ball-exercise-
       Diplo’s latest study, Mapping the challenges and opportunities of artificial intelligence for the co...
  [5] score=0.4840 | https://www.diplomacy.edu/event/diplo-at-igf2025/
       This session explored critical questions shaping the future of global AI governance....
  [6] score=0.4134 | https://www.diplomacy.edu/the-diplo

/root/.cache/pypoetry/virtualenvs/diplomacy-edu-chatbot-api-OinSWiOQ-py3.12/lib/python3.12/site-packages/starlette/config.py:60: UserWarning: Config file '.env' not found.
  warnings.warn(f"Config file '{env_file}' not found.")


## 2. Grupisanje po URL + Sekciji

Recenice se grupisu po `(url, section)` i agregatni skor se racuna.

In [6]:
from collections import defaultdict

groups = defaultdict(lambda: {"sentences": [], "scores": [], "title": "", "label": ""})

for obj in results.objects:
    p = obj.properties
    score = obj.metadata.score if obj.metadata else 0
    url = p.get('link', '')
    section = p.get('section', 'general')
    key = (url, section)
    groups[key]["sentences"].append({
        "text": p.get('sentence', ''),
        "score": score,
    })
    groups[key]["scores"].append(score)
    groups[key]["title"] = p.get('h1', '') or p.get('last_h_title', '')
    groups[key]["label"] = p.get('post_type', '')
    groups[key]["url"] = url
    groups[key]["parent_document_hash"] = p.get('parent_document_hash', '')

# Agregiraj i sortiraj
sorted_groups = sorted(groups.items(), key=lambda x: max(x[1]["scores"]), reverse=True)

print(f"{len(sorted_groups)} grupa (url, section)\n")
for i, ((url, section), g) in enumerate(sorted_groups[:8]):
    top_score = max(g['scores'])
    n = len(g['sentences'])
    title = g['title'][:50]
    print(f"  [{i+1}] score={top_score:.4f} | {n} sent | {title}")
    print(f"       URL: {url[:70]}")
    print(f"       Section: {section}")
    print(f"       Hash: {g.get('parent_document_hash', '(nema)')}")
    print()

26 grupa (url, section)

  [1] score=0.7500 | 1 sent | Diplo Knowledge Ecology: Where human wisdom meets 
       URL: https://www.diplomacy.edu/diplo-ai-agents/
       Section: 5_governance_and_policy
       Hash: None

  [2] score=0.6478 | 1 sent | Issue 479 – 15 November 2023
       URL: https://www.diplomacy.edu/diplo-news/issue479/
       Section: byte_sized_insights_5
       Hash: None

  [3] score=0.5103 | 1 sent | Diplo Knowledge Ecology: Where human wisdom meets 
       URL: https://www.diplomacy.edu/diplo-ai-agents/
       Section: general
       Hash: None

  [4] score=0.4918 | 1 sent | [Briefing #51] Internet governance forecast for 20
       URL: https://www.diplomacy.edu/blog/diplos-crystal-ball-exercise-digital-po
       Section: ai_governance
       Hash: None

  [5] score=0.4840 | 4 sent | Diplo/GIP at IGF 2025
       URL: https://www.diplomacy.edu/event/diplo-at-igf2025/
       Section: wednesday_25_june_1130_1300_cest_0930_1100_utc_plenary_hall_event_link_on_igf_we
  

## 3. Neo4j: Lookup dokumenta po hash-u

Proveri da li se retrieved URL-ovi nalaze u Neo4j grafu.

In [7]:
import hashlib

def md5(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()

def url_variants(url: str) -> list[str]:
    base = {url}
    if "://www." in url:
        base.add(url.replace("://www.", "://"))
    else:
        base.add(url.replace("://", "://www."))
    expanded = set()
    for u in base:
        expanded.add(u)
        expanded.add(u.rstrip("/") + "/")
        expanded.add(u.rstrip("/"))
    return list(expanded)

# Probaj top 5 URL-ova
unique_urls = list(dict.fromkeys(url for (url, _), _ in sorted_groups))[:5]

for url in unique_urls:
    variants = url_variants(url)
    hashes = [md5(v) for v in variants]
    
    # Proveri u Neo4j
    doc = None
    for h in hashes:
        doc = await neo4j_client.get_document_by_hash(h, NEO4J_DATABASE)
        if doc:
            break
    
    if not doc:
        doc = await neo4j_client.get_document_by_url(url, NEO4J_DATABASE)
    
    status = f"FOUND (hash={doc.document_hash[:16]}...)" if doc else "NOT IN GRAPH"
    print(f"  {status} | {url[:70]}")
    if doc:
        print(f"    Name: {doc.name[:60]}")
        print(f"    Labels: {doc.labels}")
    print()

  NOT IN GRAPH | https://www.diplomacy.edu/diplo-ai-agents/

  NOT IN GRAPH | https://www.diplomacy.edu/diplo-news/issue479/

  FOUND (hash=2085511381563ef0...) | https://www.diplomacy.edu/blog/diplos-crystal-ball-exercise-digital-po
    Name: [Briefing #51] Internet governance forecast for 2019
    Labels: ['Document', 'Blog']

  FOUND (hash=1802fbbc6962d307...) | https://www.diplomacy.edu/event/diplo-at-igf2025/
    Name: Diplo/GIP at IGF 2025
    Labels: ['Document', 'Event']

  NOT IN GRAPH | https://www.diplomacy.edu/the-diplo-ai-ecosystem/



## 4. Neo4j: Direktne relacije jednog dokumenta

Izaberi jedan dokument i pogledaj sve njegove outgoing relacije.

In [8]:
# Izaberi URL za istrazivanje (promeni po zelji)
EXPLORE_URL = unique_urls[2]  # ili hardkodiraj URL

# Nadji hash
explore_doc = None
for h in [md5(v) for v in url_variants(EXPLORE_URL)]:
    explore_doc = await neo4j_client.get_document_by_hash(h, NEO4J_DATABASE)
    if explore_doc:
        break
if not explore_doc:
    explore_doc = await neo4j_client.get_document_by_url(EXPLORE_URL, NEO4J_DATABASE)

if not explore_doc:
    print(f"Dokument nije pronadjen u Neo4j: {EXPLORE_URL}")
else:
    print(f"Dokument: {explore_doc.name}")
    print(f"Hash: {explore_doc.document_hash}")
    print(f"Labels: {explore_doc.labels}")
    print(f"Properties: {explore_doc.properties}")

Dokument: [Briefing #51] Internet governance forecast for 2019
Hash: 2085511381563ef01b682ca79e548bf1
Labels: ['Document', 'Blog']
Properties: {'date': '2019-02-05 10:56:30', 'wp_id': '2186', 'site': 'diplomacy.edu', 'name': '[Briefing #51] Internet governance forecast for 2019', 'post_type': 'blog', 'text': "Populism is spreading. The global economy is more fragile. A trade war between the USA and China is under way. And tech companies are arousing growing angst. This is how The Economist has described the start of 2019.\n\n\nThe digital realm is mirroring these societal developments. Cyber risks and uncertainties are growing. A cyber-arms race is in the making. Opportunity-wise, artificial intelligence (AI), quantum computing, and blockchain are spearheading a wave of new applications in health, agriculture, and development.\n\n\nIn 2019, the digital realpolitik trend will accelerate. A ‘digital G2’ with China and the USA is emerging with all leading digital companies being based in 

In [9]:
if explore_doc:
    doc_hash = explore_doc.document_hash
    
    # Direktne (outgoing) relacije
    direct = await neo4j_client.get_direct_relations(doc_hash, NEO4J_DATABASE, limit=50)
    print(f"Outgoing relacije: {len(direct)}\n")
    
    by_type = defaultdict(list)
    for r in direct:
        by_type[r.relationship].append(r)
    
    for rel_type, rels in sorted(by_type.items()):
        print(f"  {rel_type} ({len(rels)}):")
        for r in rels[:5]:
            labels = [l for l in r.target_labels if l != 'Document']
            print(f"    -> {r.target_name[:50]} {labels}")
        if len(rels) > 5:
            print(f"    ... i jos {len(rels)-5}")
        print()

Outgoing relacije: 4

  PUBLISHED_ON (1):
    -> February 2019 ['Date']

  RELATED_BLOG_&_TOPICS (2):
    -> Artificial Intelligence ['Topic']
    -> Gender rights online ['Topic']

  TAGGED_WITH (1):
    -> Diplo Blog ['Tag']



In [10]:
if explore_doc:
    # Incoming relacije (ko referencira ovaj dokument)
    incoming = await neo4j_client.get_incoming_relations(doc_hash, NEO4J_DATABASE)
    print(f"Incoming relacije: {len(incoming)}\n")
    for r in incoming[:10]:
        labels = [l for l in r.target_labels if l != 'Document']
        print(f"  {r.source_name[:50]} --[{r.relationship}]--> ovaj doc  {labels}")

Incoming relacije: 0



In [11]:
if explore_doc:
    # Topic hijerarhija
    hierarchy = await neo4j_client.get_topic_hierarchy(doc_hash, NEO4J_DATABASE)
    print(f"Topic hierarchy ({len(hierarchy)} topics):\n")
    for h in hierarchy:
        chain = h.get('hierarchy', [])
        topic = h.get('topic', '')
        if chain:
            print(f"  {topic}: {' -> '.join(chain)}")
        else:
            print(f"  {topic}: (no parent)")

Topic hierarchy (4 topics):

  Artificial Intelligence: Artificial Intelligence -> Infrastructure
  Artificial Intelligence: Artificial Intelligence -> Infrastructure -> Internet governance and digital policy
  Gender rights online: Gender rights online -> Human rights
  Gender rights online: Gender rights online -> Human rights -> Internet governance and digital policy


## 5. Full Graph Expansion (kao u production-u)

Pokrece `GraphExpansionService.expand_documents()` — isti kod koji koristi LangGraph node.

In [12]:
import app.core.neo4j_config as _neo4j_cfg
_neo4j_cfg.GRAPH_EXPANSION_ENABLED = True

from app.ai.ai_services.graph_expansion import GraphExpansionService
import app.ai.ai_services.graph_expansion as _ge_mod
_ge_mod.GRAPH_EXPANSION_ENABLED = True

from langchain_core.documents import Document

graph_service = GraphExpansionService(neo4j_client)

# Napravi Document objekte od top grupa (simulira retriever output)
fake_docs = []
for (url, section), g in sorted_groups[:8]:
    content = " ".join(s["text"][:200] for s in g["sentences"][:3])
    fake_docs.append(Document(
        page_content=content,
        metadata={
            "url": url,
            "title": g["title"],
            "parent_document_hash": g.get("parent_document_hash", ""),
        }
    ))

print(f"Expanding {len(fake_docs)} docs...\n")
expansions, elapsed = await graph_service.expand_documents(fake_docs, site="diplomacy.edu")
print(f"Done in {elapsed:.3f}s — {len(expansions)} docs expanded\n")

for url, exp in expansions.items():
    print(f"--- {exp.document_name[:60]} ---")
    print(f"  Hash: {exp.document_hash}")
    if exp.topics: print(f"  Topics: {exp.topics}")
    if exp.subtopic_of: print(f"  Hierarchy: {exp.subtopic_of}")
    if exp.people: print(f"  People: {exp.people}")
    if exp.actors: print(f"  Actors: {exp.actors}")
    if exp.tags: print(f"  Tags: {exp.tags[:8]}")
    if exp.related_documents:
        print(f"  Related ({len(exp.related_documents)}):")
        for rd in exp.related_documents[:5]:
            lbls = [l for l in rd.labels if l != 'Document']
            print(f"    [{lbls[0] if lbls else '?'}] {rd.name[:50]}")
    print(f"  Relations total: {len(exp.relations)}")
    print()

Expanding 8 docs...



Done in 0.278s — 3 docs expanded

--- [Briefing #51] Internet governance forecast for 2019 ---
  Hash: 2085511381563ef01b682ca79e548bf1
  Topics: ['Artificial Intelligence', 'Gender rights online']
  Hierarchy: ['Artificial Intelligence → Infrastructure', 'Artificial Intelligence → Infrastructure → Internet governance and digital policy', 'Gender rights online → Human rights', 'Gender rights online → Human rights → Internet governance and digital policy']
  Tags: ['Diplo Blog']
  Relations total: 4

--- Diplo/GIP at IGF 2025 ---
  Hash: 1802fbbc6962d307678700f5ffa55093
  Topics: ['Internet governance and digital policy', 'Sustainable development', 'Digital standards', 'Artificial Intelligence', 'Critical internet resources']
  Hierarchy: ['Sustainable development → Development', 'Sustainable development → Development → Internet governance and digital policy', 'Digital standards → Infrastructure', 'Digital standards → Infrastructure → Internet governance and digital policy', 'Artificial

## 6. Formatiran kontekst za LLM

Ovo je tekst koji se dodaje u ToolMessage posle retrieval rezultata.

In [13]:
graph_context = graph_service.format_graph_context(fake_docs, expansions)
if graph_context:
    print(graph_context)
else:
    print("(prazan kontekst — nijedan doc nema graph relacije)")



--- KNOWLEDGE GRAPH CONTEXT ---

Graph context for "[Briefing #51] Internet governance forecast for 2019":
  Topics: Artificial Intelligence, Gender rights online
  Topic hierarchy: Artificial Intelligence → Infrastructure; Artificial Intelligence → Infrastructure → Internet governance and digital policy; Gender rights online → Human rights; Gender rights online → Human rights → Internet governance and digital policy
  Tags: Diplo Blog

Graph context for "Diplo/GIP at IGF 2025":
  Topics: Internet governance and digital policy, Sustainable development, Digital standards, Artificial Intelligence, Critical internet resources
  Topic hierarchy: Sustainable development → Development; Sustainable development → Development → Internet governance and digital policy; Digital standards → Infrastructure; Digital standards → Infrastructure → Internet governance and digital policy; Artificial Intelligence → Infrastructure; Artificial Intelligence → Infrastructure → Internet governance and digital

## 7. Enriched metadata

Pogledaj sta se dodaje u `Document.metadata` za svaki expanded doc.

In [14]:
enriched = graph_service.enrich_document_metadata(fake_docs, expansions)

for doc in enriched[:5]:
    m = doc.metadata
    expanded = m.get('_graph_expanded', False)
    print(f"{'[EXPANDED]' if expanded else '[plain]':12s} {m.get('title', '')[:50]}")
    if expanded:
        print(f"  _document_hash: {m.get('_document_hash', '')}")
        print(f"  _graph_topics: {m.get('_graph_topics', [])}")
        print(f"  _graph_people: {m.get('_graph_people', [])}")
        print(f"  _graph_actors: {m.get('_graph_actors', [])}")
        print(f"  _graph_tags: {m.get('_graph_tags', [])[:5]}")
        print(f"  _graph_related_count: {m.get('_graph_related_count', 0)}")
    print()

[plain]      Diplo Knowledge Ecology: Where human wisdom meets 

[plain]      Issue 479 – 15 November 2023

[plain]      Diplo Knowledge Ecology: Where human wisdom meets 

[EXPANDED]   [Briefing #51] Internet governance forecast for 20
  _document_hash: 2085511381563ef01b682ca79e548bf1
  _graph_topics: ['Artificial Intelligence', 'Gender rights online']
  _graph_people: []
  _graph_actors: []
  _graph_tags: ['Diplo Blog']
  _graph_related_count: 0

[EXPANDED]   Diplo/GIP at IGF 2025
  _document_hash: 1802fbbc6962d307678700f5ffa55093
  _graph_topics: ['Internet governance and digital policy', 'Sustainable development', 'Digital standards', 'Artificial Intelligence', 'Critical internet resources']
  _graph_people: []
  _graph_actors: ['Internet Governance Forum']
  _graph_tags: ['IGF. Internet Governance Forum', 'Infrastructure', 'Development']
  _graph_related_count: 0



## 8. Raw Cypher upiti

Pokreni proizvoljne Cypher upite direktno na Neo4j.

In [15]:
# ── Proizvoljni Cypher upit ──
CYPHER = """
MATCH (d:Document)-[r]->(t:Topic)
WHERE type(r) CONTAINS 'TOPICS'
AND t.name CONTAINS 'Artificial Intelligence'
RETURN d.name AS doc_name, d.url AS url, type(r) AS rel, t.name AS topic
LIMIT 15
"""

await neo4j_client.connect()
async with neo4j_client._driver.session(database=NEO4J_DATABASE) as session:
    result = await session.run(CYPHER)
    records = [r async for r in result]

print(f"{len(records)} results\n")
for r in records:
    print(f"  [{r['rel']}] {r['doc_name'][:50]}")
    print(f"    Topic: {r['topic']}")
    print(f"    URL: {(r['url'] or '')[:60]}")
    print()

15 results

  [RELATED_BLOG_&_TOPICS] The impact of AI on human impulsivity and health
    Topic: Artificial Intelligence
    URL: https://www.diplomacy.edu/blog/the-impact-of-artificial-inte

  [RELATED_BLOG_&_TOPICS] Do we really need frontier AI for everyday work?
    Topic: Artificial Intelligence
    URL: https://www.diplomacy.edu/blog/do-we-really-need-frontier-ai

  [RELATED_BLOG_&_TOPICS] The fading of human agency in automated systems
    Topic: Artificial Intelligence
    URL: https://www.diplomacy.edu/blog/the-fading-of-human-agency-in

  [RELATED_BLOG_&_TOPICS] Beyond answers: How AI is redefining web communica
    Topic: Artificial Intelligence
    URL: https://www.diplomacy.edu/blog/beyond-answers-how-ai-is-rede

  [RELATED_BLOG_&_TOPICS] Automated exclusion: The crisis of inaccessible AI
    Topic: Artificial Intelligence
    URL: https://www.diplomacy.edu/blog/automated-exclusion-crisis-of

  [RELATED_BLOG_&_TOPICS] Digital ghosts
    Topic: Artificial Intelligence
    

In [16]:
# ── Graf statistike ──
STATS_CYPHER = """
MATCH (n)
RETURN labels(n)[0] AS label, count(*) AS count
ORDER BY count DESC
LIMIT 20
"""

async with neo4j_client._driver.session(database=NEO4J_DATABASE) as session:
    result = await session.run(STATS_CYPHER)
    records = [r async for r in result]

print("Neo4j graf statistike:\n")
for r in records:
    print(f"  {r['label']:20s} {r['count']:>6}")

Neo4j graf statistike:

  Document               8986
  Tag                    4673
  Date                    272


## 9. A/B Comparison: Baseline vs Graph-Enriched RAG

End-to-end poređenje:
- **Baseline**: Weaviate retrieval → LLM (bez graph konteksta)
- **Graph-Enriched**: Weaviate retrieval → Neo4j expansion → LLM (sa graph kontekstom)

Meri se: retrieval vreme, graph expansion vreme, LLM vreme, ukupno vreme.
Na kraju LLM-as-Judge ocenjuje koji je odgovor bolji.

In [17]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Svi sekreti se učitavaju iz ../.env — ne hardkoduj ih ovde.
LLM_URL = _cfg("LOCAL_LLM_URL", cast=str)
LLM_MODEL = _cfg("LOCAL_LLM_MODEL", cast=str)
LLM_KEY = _cfg("LOCAL_LLM_KEY", cast=str)

llm = ChatOpenAI(
    model_name=LLM_MODEL,
    api_key=LLM_KEY,
    base_url=LLM_URL,
    temperature=0.3,
)

judge_llm = ChatOpenAI(
    model_name=LLM_MODEL,
    api_key=LLM_KEY,
    base_url=LLM_URL,
    temperature=0.0,
)

resp = await llm.ainvoke([HumanMessage(content="Say 'LLM ready' in 3 words.")])
print(f"LLM check: {resp.content.strip()}")

LLM check: LLM is ready.


In [18]:
import json, random
from pathlib import Path
from pymongo import MongoClient

# Svi sekreti se učitavaju iz ../.env — ne hardkoduj ih ovde.
MONGO_URI = _cfg("DB_CONNECTION", cast=str)
MONGO_DB = _cfg("BENCHMARK_MONGO_DB", cast=str, default="chatbot_humainism_ai")

_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
_db = _client[MONGO_DB]
_raw = list(_db.messages.find().sort("timestamp", -1))
print(f"MongoDB: {len(_raw)} messages loaded from production\n")

_seen = {}
for m in _raw:
    q = m.get("message", "").strip()
    key = q.lower()
    if key and len(q) > 10 and key not in _seen:
        _seen[key] = q
ALL_PROD_QUESTIONS = list(_seen.values())
print(f"Unique questions (>10 chars): {len(ALL_PROD_QUESTIONS)}")

N_BENCHMARK = min(100, len(ALL_PROD_QUESTIONS))
random.seed(42)
TEST_QUESTIONS = random.sample(ALL_PROD_QUESTIONS, N_BENCHMARK)

print(f"\nBenchmark pitanja ({len(TEST_QUESTIONS)}):")
for i, q in enumerate(TEST_QUESTIONS):
    print(f"  {i+1:3d}. {q[:90]}")

_client.close()

5 test pitanja definisano.


In [19]:
from weaviate.classes.query import HybridFusion, Filter
from collections import defaultdict
from dataclasses import dataclass, field

SYSTEM_PROMPT = (
    "You are Diplorene, a helpful AI assistant specializing in diplomacy, "
    "internet governance, and digital policy. Answer based ONLY on the provided "
    "context. Cite sources as [1], [2], etc. If the context doesn't contain "
    "enough information, say so honestly."
)

@dataclass
class ABResult:
    question: str
    # baseline
    baseline_answer: str = ""
    baseline_sources: list = field(default_factory=list)
    t_baseline_retrieval: float = 0.0
    t_baseline_llm: float = 0.0
    t_baseline_total: float = 0.0
    # graph-enriched
    graph_answer: str = ""
    graph_sources: list = field(default_factory=list)
    graph_context_text: str = ""
    t_graph_retrieval: float = 0.0
    t_graph_expansion: float = 0.0
    t_graph_llm: float = 0.0
    t_graph_total: float = 0.0
    n_expanded: int = 0
    # judge
    judge_verdict: str = ""
    judge_reasoning: str = ""
    judge_scores: dict = field(default_factory=dict)


async def retrieve_and_group(question: str, limit: int = 30):
    """Hybrid search → group by (url, section) → return top groups as Documents."""
    qv = embeddings.embed_query(question)
    col = wv_client.collections.get("DiploChunk_contextual")

    res = col.query.hybrid(
        query=question,
        vector=qv,
        alpha=0.75,
        fusion_type=HybridFusion.RELATIVE_SCORE,
        limit=limit,
        filters=Filter.by_property("chunk_level").equal("sentence")
            & Filter.by_property("visibility").not_equal("private"),
        query_properties=["sentence"],
        return_metadata=["score"],
    )

    groups = defaultdict(lambda: {"sentences": [], "scores": [], "title": "", "url": ""})
    for obj in res.objects:
        p = obj.properties
        score = obj.metadata.score if obj.metadata else 0
        url = p.get('link', '')
        section = p.get('section', 'general')
        key = (url, section)
        groups[key]["sentences"].append({"text": p.get('sentence', ''), "score": score})
        groups[key]["scores"].append(score)
        groups[key]["title"] = p.get('h1', '') or p.get('last_h_title', '')
        groups[key]["url"] = url

    sorted_g = sorted(groups.items(), key=lambda x: max(x[1]["scores"]), reverse=True)

    docs = []
    for (url, section), g in sorted_g[:8]:
        content = " ".join(s["text"] for s in g["sentences"][:5])
        docs.append(Document(
            page_content=content,
            metadata={"url": url, "title": g["title"], "parent_document_hash": ""},
        ))
    return docs


def format_context_for_llm(docs, graph_context=""):
    """Format retrieved docs (+ optional graph context) into a prompt string."""
    parts = []
    sources = []
    for i, doc in enumerate(docs):
        title = doc.metadata.get('title', 'Unknown')
        url = doc.metadata.get('url', '')
        parts.append(f"[{i+1}] {title}\n{doc.page_content}")
        sources.append({"idx": i+1, "title": title, "url": url})
    context = "\n\n---\n\n".join(parts)
    if graph_context:
        context += graph_context
    return context, sources


print("Helper functions defined.")

Helper functions defined.


In [20]:
ab_results: list[ABResult] = []

await neo4j_client.connect()

for qi, question in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*80}")
    print(f"  Q{qi+1}: {question}")
    print(f"{'='*80}")
    r = ABResult(question=question)

    # ── BASELINE: retrieval → LLM (no graph) ──
    t0 = time.time()
    docs = await retrieve_and_group(question)
    r.t_baseline_retrieval = time.time() - t0

    context_text, r.baseline_sources = format_context_for_llm(docs)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context_text}\n\nQuestion: {question}"),
    ]

    t0 = time.time()
    resp = await llm.ainvoke(messages)
    r.t_baseline_llm = time.time() - t0
    r.baseline_answer = resp.content.strip()
    r.t_baseline_total = r.t_baseline_retrieval + r.t_baseline_llm

    print(f"\n  [BASELINE]  retrieval={r.t_baseline_retrieval:.2f}s  llm={r.t_baseline_llm:.2f}s  total={r.t_baseline_total:.2f}s")
    print(f"  Answer ({len(r.baseline_answer)} chars): {r.baseline_answer[:200]}...")

    # ── GRAPH-ENRICHED: retrieval → graph expansion → LLM ──
    t0 = time.time()
    docs_g = await retrieve_and_group(question)
    r.t_graph_retrieval = time.time() - t0

    t0 = time.time()
    expansions, _ = await graph_service.expand_documents(docs_g, site="diplomacy.edu")
    r.t_graph_expansion = time.time() - t0
    r.n_expanded = len(expansions)

    graph_ctx = graph_service.format_graph_context(docs_g, expansions)
    r.graph_context_text = graph_ctx
    context_text_g, r.graph_sources = format_context_for_llm(docs_g, graph_context=graph_ctx)

    messages_g = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context_text_g}\n\nQuestion: {question}"),
    ]

    t0 = time.time()
    resp_g = await llm.ainvoke(messages_g)
    r.t_graph_llm = time.time() - t0
    r.graph_answer = resp_g.content.strip()
    r.t_graph_total = r.t_graph_retrieval + r.t_graph_expansion + r.t_graph_llm

    print(f"\n  [GRAPH]     retrieval={r.t_graph_retrieval:.2f}s  expansion={r.t_graph_expansion:.2f}s  llm={r.t_graph_llm:.2f}s  total={r.t_graph_total:.2f}s")
    print(f"  Expanded: {r.n_expanded} docs | Graph context: {len(graph_ctx)} chars")
    print(f"  Answer ({len(r.graph_answer)} chars): {r.graph_answer[:200]}...")

    ab_results.append(r)

print(f"\n\nDone: {len(ab_results)} questions processed.")


  Q1: What is AI governance and how does Diplo approach it?

  [BASELINE]  retrieval=0.10s  llm=3.57s  total=3.67s
  Answer (2106 chars): **AI governance** is the collection of policies, standards, institutions and processes that steer how artificial‑intelligence systems are designed, built, deployed and used.  In the Diplo literature i...

  [GRAPH]     retrieval=0.10s  expansion=0.11s  llm=4.28s  total=4.50s
  Expanded: 3 docs | Graph context: 1259 chars
  Answer (2554 chars): **AI governance** is the set of policies, norms, institutions and technical safeguards that steer the whole life‑cycle of artificial‑intelligence systems – from the chips that run them, through the da...

  Q2: What happened at IGF 2025 regarding internet governance?

  [BASELINE]  retrieval=0.08s  llm=3.11s  total=3.19s
  Answer (2272 chars): The Internet Governance Forum (IGF) convened in Lillestrøm, Norway in 2025 and served as a hub for a series of high‑level, multi‑stakeholder sessions that tackled the mo

### LLM-as-Judge

Za svaki par odgovora, nezavisni LLM sudija ocenjuje:
- **Completeness** (1-5): koliko potpuno odgovara na pitanje
- **Accuracy** (1-5): koristi li samo informacije iz konteksta
- **Specificity** (1-5): konkretni detalji vs genericke izjave
- **Source usage** (1-5): koliko dobro citira izvore

Sudija ne zna koji je "A" a koji "B" (randomizovani redosled).

In [21]:
import json, random

JUDGE_PROMPT = """You are an expert evaluator comparing two answers to the same question.
Both answers were generated by a RAG system using retrieved documents as context.

Question: {question}

=== Answer A ===
{answer_a}

=== Answer B ===
{answer_b}

Evaluate EACH answer on these criteria (1-5 scale):
1. **Completeness**: How thoroughly does it address all aspects of the question?
2. **Accuracy**: Does it stick to information from the provided context without hallucinating?
3. **Specificity**: Does it provide concrete details, names, dates, examples (vs vague generalities)?
4. **Source_usage**: How well does it cite and reference sources?

Then decide an overall **winner**: "A", "B", or "TIE".

Respond ONLY with valid JSON (no markdown, no explanation outside JSON):
{{
  "scores_a": {{"completeness": N, "accuracy": N, "specificity": N, "source_usage": N}},
  "scores_b": {{"completeness": N, "accuracy": N, "specificity": N, "source_usage": N}},
  "winner": "A" or "B" or "TIE",
  "reasoning": "1-2 sentence explanation"
}}"""


async def judge_pair(question: str, baseline_answer: str, graph_answer: str) -> dict:
    """Run LLM-as-Judge on a pair, randomizing A/B assignment."""
    swap = random.random() > 0.5
    if swap:
        a_answer, b_answer = graph_answer, baseline_answer
        a_label, b_label = "graph", "baseline"
    else:
        a_answer, b_answer = baseline_answer, graph_answer
        a_label, b_label = "baseline", "graph"

    prompt = JUDGE_PROMPT.format(
        question=question,
        answer_a=a_answer,
        answer_b=b_answer,
    )

    resp = await judge_llm.ainvoke([HumanMessage(content=prompt)])
    raw = resp.content.strip()

    # parse JSON (handle markdown fences)
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        return {"error": raw[:300], "swap": swap}

    # un-swap so scores map back to baseline/graph
    if swap:
        baseline_scores = result.get("scores_b", {})
        graph_scores = result.get("scores_a", {})
        raw_winner = result.get("winner", "TIE")
        winner_map = {"A": "graph", "B": "baseline", "TIE": "TIE"}
    else:
        baseline_scores = result.get("scores_a", {})
        graph_scores = result.get("scores_b", {})
        raw_winner = result.get("winner", "TIE")
        winner_map = {"A": "baseline", "B": "graph", "TIE": "TIE"}

    return {
        "baseline_scores": baseline_scores,
        "graph_scores": graph_scores,
        "winner": winner_map.get(raw_winner, "TIE"),
        "reasoning": result.get("reasoning", ""),
        "swap": swap,
    }


print("Judge function defined.")

Judge function defined.


In [22]:
for i, r in enumerate(ab_results):
    print(f"\nJudging Q{i+1}: {r.question[:60]}...")
    verdict = await judge_pair(r.question, r.baseline_answer, r.graph_answer)

    if "error" in verdict:
        print(f"  Judge parse error: {verdict['error'][:100]}")
        r.judge_verdict = "ERROR"
        r.judge_reasoning = verdict.get("error", "")
        continue

    r.judge_scores = {
        "baseline": verdict["baseline_scores"],
        "graph": verdict["graph_scores"],
    }
    r.judge_verdict = verdict["winner"]
    r.judge_reasoning = verdict["reasoning"]

    bs = verdict["baseline_scores"]
    gs = verdict["graph_scores"]
    b_avg = sum(bs.values()) / max(len(bs), 1)
    g_avg = sum(gs.values()) / max(len(gs), 1)

    print(f"  Baseline: {bs}  avg={b_avg:.1f}")
    print(f"  Graph:    {gs}  avg={g_avg:.1f}")
    print(f"  Winner:   {verdict['winner'].upper()}")
    print(f"  Reason:   {verdict['reasoning']}")

print("\nAll judging complete.")


Judging Q1: What is AI governance and how does Diplo approach it?...
  Baseline: {'completeness': 4, 'accuracy': 5, 'specificity': 4, 'source_usage': 4}  avg=4.2
  Graph:    {'completeness': 5, 'accuracy': 5, 'specificity': 5, 'source_usage': 5}  avg=5.0
  Winner:   GRAPH
  Reason:   Answer B covers more facets of Diplo's approach with a structured table and additional details, while maintaining accurate, well‑cited information.

Judging Q2: What happened at IGF 2025 regarding internet governance?...
  Baseline: {'completeness': 5, 'accuracy': 5, 'specificity': 4, 'source_usage': 5}  avg=4.8
  Graph:    {'completeness': 5, 'accuracy': 5, 'specificity': 4, 'source_usage': 5}  avg=4.8
  Winner:   GRAPH
  Reason:   Both answers are equally thorough and accurate, but Answer A presents the information slightly more cohesively with distinct bullet points, giving it a marginal edge.

Judging Q3: How does Diplo use AI in its educational programs?...
  Baseline: {'completeness': 5, 'accuracy':

### Summary Table

In [23]:
criteria = ["completeness", "accuracy", "specificity", "source_usage"]

# ── Per-question table ──
header = f"{'#':>2} {'Question':<45} {'B_time':>6} {'G_time':>6} {'Δ_exp':>6} │ "
header += " ".join(f"B_{c[:4]:>4}" for c in criteria) + "  "
header += " ".join(f"G_{c[:4]:>4}" for c in criteria) + "  Winner"
print(header)
print("─" * len(header))

wins = {"baseline": 0, "graph": 0, "TIE": 0, "ERROR": 0}
total_b_scores = {c: 0 for c in criteria}
total_g_scores = {c: 0 for c in criteria}
n_judged = 0

for i, r in enumerate(ab_results):
    q_short = r.question[:43]
    bt = f"{r.t_baseline_total:.1f}s"
    gt = f"{r.t_graph_total:.1f}s"
    exp = f"+{r.t_graph_expansion:.2f}s"

    bs = r.judge_scores.get("baseline", {})
    gs = r.judge_scores.get("graph", {})

    if bs and gs:
        b_vals = " ".join(f"{bs.get(c, 0):>5}" for c in criteria)
        g_vals = " ".join(f"{gs.get(c, 0):>5}" for c in criteria)
        for c in criteria:
            total_b_scores[c] += bs.get(c, 0)
            total_g_scores[c] += gs.get(c, 0)
        n_judged += 1
    else:
        b_vals = "   -" * 4
        g_vals = "   -" * 4

    winner_icon = {"baseline": "◀ BASE", "graph": "GRAPH ▶", "TIE": "  TIE  ", "ERROR": " ERROR "}
    w = winner_icon.get(r.judge_verdict, "  ?  ")
    wins[r.judge_verdict] = wins.get(r.judge_verdict, 0) + 1

    print(f"{i+1:>2} {q_short:<45} {bt:>6} {gt:>6} {exp:>6} │ {b_vals}  {g_vals}  {w}")

# ── Averages ──
print("─" * len(header))
if n_judged > 0:
    avg_bt = sum(r.t_baseline_total for r in ab_results) / len(ab_results)
    avg_gt = sum(r.t_graph_total for r in ab_results) / len(ab_results)
    avg_exp = sum(r.t_graph_expansion for r in ab_results) / len(ab_results)
    b_avgs = " ".join(f"{total_b_scores[c]/n_judged:>5.1f}" for c in criteria)
    g_avgs = " ".join(f"{total_g_scores[c]/n_judged:>5.1f}" for c in criteria)
    print(f"{'AVG':>2} {'':45} {avg_bt:>5.1f}s {avg_gt:>5.1f}s +{avg_exp:>.2f}s │ {b_avgs}  {g_avgs}")

# ── Wins summary ──
print(f"\n{'='*50}")
print(f"  WINS:  Baseline={wins['baseline']}  Graph={wins['graph']}  Tie={wins['TIE']}  Error={wins['ERROR']}")
overhead = sum(r.t_graph_expansion for r in ab_results) / max(len(ab_results), 1)
print(f"  AVG GRAPH EXPANSION OVERHEAD: +{overhead:.3f}s")
print(f"{'='*50}")

 # Question                                      B_time G_time  Δ_exp │ B_comp B_accu B_spec B_sour  G_comp G_accu G_spec G_sour  Winner
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 1 What is AI governance and how does Diplo ap     3.7s   4.5s +0.11s │     4     5     4     4      5     5     5     5  GRAPH ▶
 2 What happened at IGF 2025 regarding interne     3.2s   3.8s +0.10s │     5     5     4     5      5     5     4     5  GRAPH ▶
 3 How does Diplo use AI in its educational pr     3.2s   3.1s +0.09s │     5     5     4     5      4     5     4     4  ◀ BASE
 4 What is the relationship between cybersecur     2.9s   4.3s +0.10s │     4     5     4     4      5     5     5     5  GRAPH ▶
 5 What are the main digital policy challenges     5.7s   4.7s +0.07s │     5     5     5     5      4     5     4     5  ◀ BASE
──────────────────────────────────────────────────────────────────────────────

In [24]:
from IPython.display import display, Markdown

for i, r in enumerate(ab_results):
    display(Markdown(f"---\n### Q{i+1}: {r.question}\n"))

    bs = r.judge_scores.get("baseline", {})
    gs = r.judge_scores.get("graph", {})
    b_avg = sum(bs.values()) / max(len(bs), 1) if bs else 0
    g_avg = sum(gs.values()) / max(len(gs), 1) if gs else 0

    display(Markdown(
        f"| | Baseline | Graph-Enriched |\n"
        f"|---|---|---|\n"
        f"| **Time** | {r.t_baseline_total:.2f}s | {r.t_graph_total:.2f}s (+{r.t_graph_expansion:.2f}s expansion) |\n"
        f"| **Score avg** | {b_avg:.1f}/5 | {g_avg:.1f}/5 |\n"
        f"| **Expanded docs** | - | {r.n_expanded} |\n"
        f"| **Winner** | {'**Winner**' if r.judge_verdict == 'baseline' else ''} | {'**Winner**' if r.judge_verdict == 'graph' else ''} |\n"
    ))

    display(Markdown(f"**Judge reasoning:** {r.judge_reasoning}\n"))

    display(Markdown(f"<details><summary>Baseline answer ({len(r.baseline_answer)} chars)</summary>\n\n{r.baseline_answer}\n\n</details>"))
    display(Markdown(f"<details><summary>Graph-enriched answer ({len(r.graph_answer)} chars)</summary>\n\n{r.graph_answer}\n\n</details>"))

    if r.graph_context_text:
        ctx_preview = r.graph_context_text[:500] + ("..." if len(r.graph_context_text) > 500 else "")
        display(Markdown(f"<details><summary>Graph context injected ({len(r.graph_context_text)} chars)</summary>\n\n```\n{ctx_preview}\n```\n\n</details>"))

---
### Q1: What is AI governance and how does Diplo approach it?


| | Baseline | Graph-Enriched |
|---|---|---|
| **Time** | 3.67s | 4.50s (+0.11s expansion) |
| **Score avg** | 4.2/5 | 5.0/5 |
| **Expanded docs** | - | 3 |
| **Winner** |  | **Winner** |


**Judge reasoning:** Answer B covers more facets of Diplo's approach with a structured table and additional details, while maintaining accurate, well‑cited information.


<details><summary>Baseline answer (2106 chars)</summary>

**AI governance** is the collection of policies, standards, institutions and processes that steer how artificial‑intelligence systems are designed, built, deployed and used.  In the Diplo literature it is described as a *layered* framework that spans the entire technical stack – from the physical hardware that runs models, through the data that feeds them, the algorithms that process that data, and finally the applications that deliver AI‑driven services [2].  Good AI governance seeks to ensure that these systems respect human rights, reflect diverse perspectives, operate on balanced data, and remain accountable to the societies they affect.

**How Diplo approaches AI governance**

1. **Blend of diplomatic expertise and hands‑on tech knowledge** – Diplo stresses that teaching and researching technology governance requires more than theory; it demands a deep, practical grasp of the underlying technologies [3][8].

2. **Mapping the full stack** – By breaking AI into hardware, data, algorithms and apps, Diplo’s analysts can pinpoint where governance interventions are needed and how they interrelate [2].

3. **Multistakeholder and rights‑focused framing** – Diplo’s work (e.g., the IGF 2025 session) highlights the importance of involving the private sector, civil society and governments so that AI systems uphold human rights and incorporate diverse viewpoints [5].

4. **Strategic foresight for diplomacy** – Studies such as “Mapping the challenges and opportunities of artificial intelligence for the conduct of diplomacy” examine how the rapid rollout of national AI strategies creates a competitive “race” and what that means for diplomatic practice [4].

5. **Preparing diplomats** – Before discussing governance or delivering courses, Diplo first ensures diplomats have a solid technical foundation in AI, enabling them to engage meaningfully in policy debates [8].

In sum, AI governance for Diplo is a comprehensive, rights‑based oversight of the entire AI lifecycle, pursued through a combination of technical fluency, diplomatic insight, and inclusive, multistakeholder processes.

</details>

<details><summary>Graph-enriched answer (2554 chars)</summary>

**AI governance** is the set of policies, norms, institutions and technical safeguards that steer the whole life‑cycle of artificial‑intelligence systems – from the chips that run them, through the data they ingest, the algorithms that process that data, to the applications that deliver outcomes – so that AI serves public interest, respects human rights and remains accountable and transparent.  In Diplo’s own description the “layers of AI governance” are hardware, data, algorithms and apps [2].

**How Diplo approaches AI governance**

| Aspect | Diplo’s practice |
|--------|------------------|
| **Blend of expertise** | Diplo deliberately merges “decades of expert knowledge in diplomacy and global governance” with a “deep, hands‑on grasp of the technologies shaping our world, such as AI” [3]. |
| **Technical grounding before policy** | The organization first ensures diplomats and policymakers “fully understand technology” before moving to governance or training [8]. |
| **Multistakeholder focus** | At events such as **IGF 2025**, Diplo examined how private‑sector actors, civil society and governments can together ensure AI systems “uphold human rights, reflect diverse perspectives, and are built on balanced data” [5]. |
| **Human‑rights and inclusivity lens** | Discussions at IGF 2025 also looked at the feasibility and inclusivity of emerging governance mechanisms, stressing that AI must be aligned with human‑rights standards and gender‑rights online [5][4]. |
| **Research & mapping** | Diplo’s briefing “Mapping the challenges and opportunities of artificial intelligence for the conduct of diplomacy” tracks the rapid rollout of national AI strategies and frames them as a “race to become the global leader in artificial intelligence” [4]. |
| **Strategic forums & incentives** | The organization works to make multistakeholder venues (e.g., the Internet Governance Forum) more relevant and impactful for AI governance, identifying incentives and formats that encourage broader participation [5]. |
| **Core mission** | Overall, “Much of Diplo’s work centres on the governance of AI, the internet, and overall tech developments” [1][6]. |

In short, Diplo defines AI governance as the coordinated oversight of AI’s technical layers to protect rights and ensure accountability, and it pursues this goal by (1) building deep technical literacy among diplomats, (2) leveraging its diplomatic expertise, (3) fostering inclusive, multistakeholder dialogue, and (4) producing research that maps emerging challenges and policy trends.

</details>

...

---
### Q5: What are the main digital policy challenges for developing countries?


| | Baseline | Graph-Enriched |
|---|---|---|
| **Time** | 5.67s | 4.67s (+0.07s expansion) |
| **Score avg** | 5.0/5 | 4.5/5 |
| **Expanded docs** | - | 0 |
| **Winner** | **Winner** |  |


**Judge reasoning:** Answer B covers a broader set of challenges with more concrete details and consistently cites sources, making it more complete and specific.


<details><summary>Baseline answer (3771 chars)</summary>

The discussion across the workshop notes and the draft Global Digital Compact documents points to a handful of recurring obstacles that most developing countries face when trying to design and implement effective digital policies:

| **Challenge** | **What the sources say** | **Key implications** |
|---|---|---|
| **Resource‑constrained implementation** | “There are challenges in implementing digital policies, especially for developing countries with resource constraints” [1][2] | Limited fiscal, technical and human‑resource capacity hampers the rollout of national digital strategies, regulatory reforms and infrastructure projects. |
| **Insufficient and unaffordable connectivity** | “A lack of access to affordable and reliable technologies and services remains a critical challenge… efforts should be deployed to reduce the price of ICT and broadband” [3][4][5] | High equipment and service costs, coupled with low household incomes, keep broadband penetration low (≈57 % in developing countries vs 90 % in developed ones) [3]. |
| **Persistent digital and broadband divide** | “There is still uneven growth in access to and the use of ICT… substantial continued digital and broadband divides between and within developed and developing countries” [3][7] | Rural‑urban, gender‑based and intra‑regional gaps widen inequality and limit inclusive digital transformation. |
| **Geopolitical and environmental vulnerability** | The same passages list “African countries, least‑developed countries, land‑locked developing countries, small‑island developing States, middle‑income countries, countries in situations of conflict, post‑conflict countries and countries affected by natural disasters” as especially exposed [3][4][5] | Conflict, post‑conflict reconstruction, isolation or disaster risk further strain the ability to invest in digital infrastructure and maintain services. |
| **Limited data‑hosting and digital‑sovereignty capacity** | “Most developing and least‑developed countries lack the capacity of hosting their huge data generated locally… key challenges… in the implementation of local servers and the adoption of data‑localisation policy” [6] | Weak domestic data centres, insufficient skilled workforce and limited regulatory frameworks impede the pursuit of digital sovereignty and secure data management. |
| **Broad constraints on digital‑economy participation** | “Most developing countries face many constraints in trying to benefit from the digital economy” [8] | Barriers such as inadequate digital skills, weak e‑commerce ecosystems, and limited access to finance restrict the ability to capture the economic gains of digitalisation. |

**In short, the main digital‑policy challenges for developing countries are:**

1. **Resource and capacity limitations** that restrict policy design, implementation, and maintenance.  
2. **Lack of affordable, reliable, and widespread connectivity**, leading to low internet penetration and high cost‑to‑income ratios.  
3. **Deepening digital divides**—both between developed and developing nations and within developing societies.  
4. **Specific vulnerabilities of certain groups of states** (land‑locked, island, conflict‑affected, disaster‑prone) that exacerbate infrastructure and service gaps.  
5. **Insufficient domestic data‑hosting and digital‑sovereignty capabilities**, making data‑localisation and security policies hard to enact.  
6. **General structural constraints that limit participation in the digital economy** (skills, finance, regulatory environment).  

These challenges are repeatedly highlighted in the workshop reports and the draft Global Digital Compact text, underscoring the need for coordinated, rights‑based capacity‑building and solidarity measures to address them.

</details>

<details><summary>Graph-enriched answer (2887 chars)</summary>

The sources point to a consistent set of obstacles that developing countries repeatedly cite when they try to design and roll out digital policies. The main challenges can be grouped into three inter‑related themes:

| Theme | Specific challenges mentioned in the context |
|-------|-----------------------------------------------|
| **1. Access‑affordability & infrastructure gaps** | • A “lack of access to affordable and reliable technologies and services” is repeatedly highlighted for African states, least‑developed, land‑locked, small‑island, middle‑income, conflict‑affected and disaster‑prone countries [3][4][5]. <br>• The cost of broadband and ICTs is higher relative to average household income, creating “a lack of affordable access” and widening the digital‑broadband divide [3][7]. <br>• Even where progress has been made, “uneven growth in access to and the use of information and communications technologies” persists, with 90 % Internet penetration in developed nations versus only 57 % in developing ones [3]. |
| **2. Resource and capacity constraints** | • Developing countries face “resource constraints” that hamper the implementation of digital policies [1][2]. <br>• Many lack the technical capacity to host the large volumes of data generated locally, which undermines digital‑sovereignty and data‑localisation efforts [6]. <br>• Limited human‑skill and institutional capacity makes it difficult to deploy and maintain local servers or to manage complex data‑governance regimes [6]. |
| **3. Structural and contextual barriers** | • Geopolitical and environmental conditions—conflict, post‑conflict recovery, natural disasters—further restrict the ability to build and sustain digital ecosystems [3][4][5]. <br>• High‑cost technology and the absence of “deliberate interventions…including through research and development and technology transfer on mutually agreed terms” impede the development of lower‑cost connectivity options [3]. <br>• Overall, “most developing and least developed countries” encounter “many constraints in trying to benefit from the digital economy” [8]. |

**In short, the main digital policy challenges for developing countries are:**  

1. **Insufficient, expensive, and unreliable ICT infrastructure** that leaves large portions of the population offline.  
2. **Limited financial and technical resources**, including the capacity to host data locally and to implement sovereignty‑oriented policies.  
3. **Context‑specific hurdles** such as conflict, post‑conflict recovery, and natural‑disaster vulnerability, which compound the difficulty of building resilient digital systems.  

Addressing these challenges will require coordinated interventions—affordable technology provision, capacity‑building, and targeted R&D/technology‑transfer mechanisms—consistent with the recommendations echoed across the cited documents. [3][6][7][8]

</details>

### Export Results to Markdown

In [25]:
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
out_dir = Path("notebooks/benchmark_results")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"ab_comparison_{timestamp}.md"

criteria = ["completeness", "accuracy", "specificity", "source_usage"]

lines = []
w = lines.append

w(f"# Graph RAG A/B Benchmark — {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
w(f"**LLM**: `{LLM_MODEL}` @ `{LLM_URL}`  ")
w(f"**Retrieval**: DiploChunk_contextual hybrid (α=0.75, limit=30)  ")
w(f"**Graph DB**: Neo4j `{NEO4J_DATABASE}` @ `{NEO4J_URI}`  ")
w(f"**Questions**: {len(ab_results)}\n")

# ── Summary table ──
w("## Summary\n")
w("| # | Question | B time | G time | Δ expand | B avg | G avg | Winner |")
w("|---|----------|--------|--------|----------|-------|-------|--------|")

wins = {"baseline": 0, "graph": 0, "TIE": 0, "ERROR": 0}
for i, r in enumerate(ab_results):
    bs = r.judge_scores.get("baseline", {})
    gs = r.judge_scores.get("graph", {})
    b_avg = sum(bs.values()) / max(len(bs), 1) if bs else 0
    g_avg = sum(gs.values()) / max(len(gs), 1) if gs else 0
    winner_str = {"baseline": "Baseline", "graph": "**Graph**", "TIE": "Tie", "ERROR": "Error"}.get(r.judge_verdict, "?")
    wins[r.judge_verdict] = wins.get(r.judge_verdict, 0) + 1
    w(f"| {i+1} | {r.question[:55]} | {r.t_baseline_total:.2f}s | {r.t_graph_total:.2f}s | +{r.t_graph_expansion:.2f}s | {b_avg:.1f} | {g_avg:.1f} | {winner_str} |")

avg_bt = sum(r.t_baseline_total for r in ab_results) / len(ab_results)
avg_gt = sum(r.t_graph_total for r in ab_results) / len(ab_results)
avg_exp = sum(r.t_graph_expansion for r in ab_results) / len(ab_results)
w("")
w(f"**Average times**: Baseline {avg_bt:.2f}s · Graph {avg_gt:.2f}s · Expansion overhead +{avg_exp:.3f}s  ")
w(f"**Wins**: Baseline {wins['baseline']} · Graph {wins['graph']} · Tie {wins['TIE']}  ")
w("")

# ── Detailed scores ──
w("## Detailed Scores\n")
w("| # | Criterion | Baseline | Graph |")
w("|---|-----------|----------|-------|")
for i, r in enumerate(ab_results):
    bs = r.judge_scores.get("baseline", {})
    gs = r.judge_scores.get("graph", {})
    for j, c in enumerate(criteria):
        q_col = f"**Q{i+1}** {r.question[:40]}" if j == 0 else ""
        w(f"| {q_col} | {c} | {bs.get(c, '-')} | {gs.get(c, '-')} |")
    w(f"| | **average** | **{sum(bs.values())/max(len(bs),1):.1f}** | **{sum(gs.values())/max(len(gs),1):.1f}** |")
w("")

# ── Per-question detail ──
w("## Full Answers\n")
for i, r in enumerate(ab_results):
    w(f"### Q{i+1}: {r.question}\n")

    bs = r.judge_scores.get("baseline", {})
    gs = r.judge_scores.get("graph", {})
    b_avg = sum(bs.values()) / max(len(bs), 1) if bs else 0
    g_avg = sum(gs.values()) / max(len(gs), 1) if gs else 0

    w(f"| | Baseline | Graph-Enriched |")
    w(f"|---|---|---|")
    w(f"| Time | {r.t_baseline_total:.2f}s | {r.t_graph_total:.2f}s (+{r.t_graph_expansion:.2f}s expansion) |")
    w(f"| Score avg | {b_avg:.1f}/5 | {g_avg:.1f}/5 |")
    w(f"| Expanded docs | — | {r.n_expanded} |")
    winner_label = "Baseline" if r.judge_verdict == "baseline" else "Graph" if r.judge_verdict == "graph" else "Tie"
    w(f"| **Winner** | {'**✓**' if r.judge_verdict == 'baseline' else ''} | {'**✓**' if r.judge_verdict == 'graph' else ''} |")
    w("")
    w(f"> **Judge:** {r.judge_reasoning}\n")

    w("<details><summary>Baseline answer</summary>\n")
    w(r.baseline_answer)
    w("\n</details>\n")

    w("<details><summary>Graph-enriched answer</summary>\n")
    w(r.graph_answer)
    w("\n</details>\n")

    if r.graph_context_text:
        w("<details><summary>Graph context injected</summary>\n")
        w(f"```\n{r.graph_context_text}\n```")
        w("\n</details>\n")

    w("---\n")

# ── Write file ──
out_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Benchmark results saved to: {out_path}")
print(f"  File size: {out_path.stat().st_size / 1024:.1f} KB")

Benchmark results saved to: notebooks/benchmark_results/ab_comparison_2026-03-30_1350.md
  File size: 33.5 KB


## 11. Cleanup

In [26]:
await neo4j_client.close()
wv_client.close()
print("Connections closed.")

Connections closed.
